# RSGM and the HuProSO3 flow, evaluated with the protocol of the HuProSO3 paper

The evaluation of the unconditional prior in the HuProSO3 paper (Dünkel, Salzmann, Pfaff), reproduced for our models and for the authors' released prior `id_0900`, so that the numbers can be put next to the paper's tables. The protocol is implemented in `hp/evaluation/huproso3_protocol.py` (tests: `tests/test_huproso3_protocol.py`).

| paper | what | here |
|---|---|---|
| Tab. 1 | recall / precision, **summed** geodesic distance over the joints (radians), mean [median], test and train | the same |
| | recall: 1k dataset poses vs. **100k** model samples, nearest neighbour | the same |
| | precision: 1k model samples vs. **100k** dataset poses, nearest neighbour | the same |
| Tab. 6 | the same with the MPJPE (mm) after forward kinematics | the same (the FK of the paper's code, 22 joints) |
| Fig. 10, 11 | precision and recall curves P(d < ε) | the same |
| Tab. 7 | mean log-likelihood of dataset poses, test ("Prior") and train ("Prior (Train)") | the same, 5000 poses (`num_eval_samples` of the paper's config) |

What differs from the paper, and why:
- **The data.** Our AMASS is the SMPL-X **G** release; the authors' is most likely the **N** release (see `evaluate_priors.ipynb`). So even the authors' prior `id_0900` does not reproduce the paper's numbers exactly here, and its "train" split here is *our* train data, which it has not seen. Its row shows how much the data alone moves the numbers.
- **The train split** is every 30th frame of the official train split (`AMASS_rotations_c30`, 497,707 poses): 100k random poses of it are statistically the same as 100k of all 14.8M frames, which would not fit in memory. The test split is all frames (117,679 poses).
- **Repeats.** The paper gives one number per cell. Here every number is the mean ± std over 3 independent draws of the dataset poses and of the 1k sample subsets (the 100k model samples are drawn once per model). The std is evaluation noise, not training noise.
- **Robust likelihood statistics** next to the mean (median, trimmed mean, share outside [0, 250]), because the mean can be dominated by a few broken values (`docs/rsgm_rare_pose_instability.md`).
- **A yardstick row** that the paper does not have: real poses of other subjects (100k train poses) in place of the model, on the test split. A model that reproduced the training distribution exactly would score about that.
- J: the paper's text says the distance sums over J = 21 joints; its code and the model have 19 rotations. We sum over 19 (as the code).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from hp.evaluation.data import load_reference
from hp.evaluation.huproso3_protocol import PAPER, PaperProtocolConfig, evaluate_paper_protocol
from hp.models.io import MANIFEST, load_model

# ---- parameters ----------------------------------------------------------------------
# Runs of the model zoo: a run folder, or a folder of runs (the newest finished one is taken). The flows are the
# converted copies made by rsgm_huproso_comparison.ipynb (weights unchanged).
MODELS = {
    # "RSGM c30":        dict(run="~/data/experiments/rsgm_c30", weights="weights_ema"),
    # "RSGM full":       dict(run="~/data/experiments/rsgm", weights="weights_ema"),
    "RSGM full t23":   dict(run="~/data/experiments/rsgm_tuned_t23", weights="weights_ema"),
    # "RSGM full r2t12": dict(run="~/data/experiments/rsgm_tuned_r2t12", weights="weights_ema"),
    "flow c30":        dict(run="~/data/experiments/so3_flow_c30/id_9030", weights="weights"),
    "flow authors (id_0900)": dict(run="~/data/experiments/so3_flow_authors/id_0900", weights="weights"),
}
YARDSTICK = "real train poses (other subjects)"  # a row with real poses in place of a model, on the test split
TEST_ROTATIONS = "~/data/AMASS_rotations"        # all frames
TRAIN_ROTATIONS = "~/data/AMASS_rotations_c30"   # every 30th frame of the train split
SPLIT = "original"                               # the official AMASS split
DEVICES = {"rsgm": "mps", "so3_flow": "cpu"}     # by model type
RSGM_LIKELIHOOD_STEPS = 200                      # ODE steps of RSGM's log-likelihood (~0.06 nats too high; 100: ~0.3)
PROTOCOL = PaperProtocolConfig(n_queries=1_000, n_pool=100_000, n_likelihood=5_000, repeats=3, seed=0)
REPORT_DIR = "~/data/experiments/huproso3_protocol"  # one report per model; reused while the settings are the same
RECOMPUTE = False
# ----------------------------------------------------------------------------------------

expand = lambda p: Path(p).expanduser()
REPORT_DIR = expand(REPORT_DIR)
REPORT_DIR.mkdir(parents=True, exist_ok=True)


def run_folder(path):
    path = expand(path)
    if (path / MANIFEST).is_file():
        return path
    runs = sorted(p for p in path.glob("run_[0-9][0-9][0-9][0-9]") if (p / "history.json").is_file() and (p / MANIFEST).is_file())
    return runs[-1] if runs else None

## 1. The data

In [ ]:
splits = {
    "test": load_reference("test", TEST_ROTATIONS, SPLIT),
    "train": load_reference("train", TRAIN_ROTATIONS, SPLIT),
}
for name, poses in splits.items():
    print(f"{name}: {len(poses):,} poses")

## 2. Evaluate

Every model on the same dataset poses (the draws of the dataset do not depend on the model). Takes about an hour from scratch, most of it RSGM's log-likelihood (the probability flow ODE with the exact divergence, 20-40 s per 1000 poses on mps); afterwards the saved reports are reused.

In [ ]:
def settings(name, extra=None):
    return dict(protocol=PROTOCOL.__dict__, test=str(TEST_ROTATIONS), train=str(TRAIN_ROTATIONS), split=SPLIT, **(extra or {}))


def cached(name, settings_, compute):
    path = REPORT_DIR / f"{name.replace(' ', '_').replace('(', '').replace(')', '')}.json"
    if path.is_file() and not RECOMPUTE:
        report = json.loads(path.read_text())
        if report.get("settings") == json.loads(json.dumps(settings_)):
            print(f"--- {name}: saved report {path}")
            return report
    report = compute()
    report["settings"] = settings_
    path.write_text(json.dumps(report))
    print(f"--- {name}: evaluated, saved to {path}")
    return report


reports = {}
for name, m in MODELS.items():
    run = run_folder(m["run"])
    if run is None:
        print(f"{name}: skipped, no finished run in {m['run']}")
        continue
    kind = json.loads((run / MANIFEST).read_text())["build_args"]["name"]

    def compute(run=run, kind=kind, weights=m["weights"]):
        model = load_model(run, device=DEVICES[kind], weights=weights)
        if kind == "rsgm":
            model.likelihood_steps = RSGM_LIKELIHOOD_STEPS
        return evaluate_paper_protocol(model, splits, PROTOCOL, progress=True)

    reports[name] = cached(name, settings(name, dict(run=str(run), weights=m["weights"], kind=kind,
                                                     likelihood_steps=RSGM_LIKELIHOOD_STEPS if kind == "rsgm" else None)), compute)

# the yardstick: 100k real poses of the train subjects in place of a model, on the test split only
rng = np.random.default_rng(1)
pool = splits["train"][rng.choice(len(splits["train"]), PROTOCOL.n_pool, replace=False)]
reports[YARDSTICK] = cached(YARDSTICK, settings(YARDSTICK, dict(pool_seed=1)),
                            lambda: evaluate_paper_protocol(None, {"test": splits["test"]}, PROTOCOL, generated=pool, progress=True))

## 3. Tab. 1: summed geodesic distance (radians)

mean [median] of the nearest-neighbour distances, ± the std of the mean over the repeats. Lower is better. The paper's rows are printed as in the paper.

In [ ]:
def cell(report, split, metric):
    if split not in report:
        return "-"
    r = report[split][metric]
    return f"{r['mean']['mean']:6.2f} [{r['median']['mean']:5.2f}] ±{r['mean']['std']:.2f}"


def paper_cell(values):
    return f"{values[0]:6.2f} [{values[1]:5.2f}]      "


def table(kind, unit, scale=1.0):
    head = f"{'':34s}" + "".join(f"{s + ' ' + m:>23s}" for s in ("test", "train") for m in ("recall", "precision"))
    print(head)
    for name, r in PAPER[kind].items():
        print(f"{'paper: ' + name:34s}" + "".join(f"{paper_cell(r[s][m]):>23s}" for s in ("test", "train") for m in ("recall", "precision")))
    print("-" * len(head))
    for name, r in reports.items():
        print(f"{name:34.33s}" + "".join(f"{cell(r, s, f'{m}_{unit}'):>23s}" for s in ("test", "train") for m in ("recall", "precision")))


table("geodesic_rad", "geodesic")

## 4. Tab. 6: MPJPE after forward kinematics (mm)

In [ ]:
table("mpjpe_mm", "mpjpe")

## 5. Tab. 7: log-likelihood

Mean log-likelihood of 5000 dataset poses (relative to the uniform distribution on SO(3)^19, higher is better), ± the std over the repeats, next to the robust statistics. A mean far from the median, or a non-zero implausible share (per-pose values outside [0, 250]), means that a few broken values carry the mean.

In [ ]:
print(f"{'':28s} {'split':6s} {'mean':>16s} {'median':>8s} {'trimmed':>8s} {'min':>9s} {'max':>9s} {'implausible':>12s}")
for name, values in PAPER["log_likelihood"].items():
    for split, v in values.items():
        print(f"{'paper: ' + name:28s} {split:6s} {v:16.2f}")
print("-" * 104)
for name, r in reports.items():
    for split in ("test", "train"):
        ll = r.get(split, {}).get("likelihood")
        if ll is None:
            continue
        print(f"{name:28.27s} {split:6s} {ll['mean']['mean']:9.2f} ±{ll['mean']['std']:5.2f} {ll['median']['mean']:8.2f} {ll['trimmed_mean']['mean']:8.2f} "
              f"{ll['min']['mean']:9.1f} {ll['max']['mean']:9.1f} {ll['implausible_share']['mean']:12.4f}")

## 6. Fig. 10 and 11: precision and recall curves

P(distance < ε) of the first repeat; higher is better. Top: summed geodesic distance (radians), bottom: MPJPE (mm).

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharey=True)
for row, (unit, label, xmax) in enumerate([("geodesic", "summed geodesic distance ε [rad]", 10), ("mpjpe", "MPJPE ε [mm]", 200)]):
    for col, (split, metric) in enumerate([(s, m) for s in ("test", "train") for m in ("recall", "precision")]):
        ax = axes[row, col]
        for k, (name, r) in enumerate(reports.items()):
            if split not in r:
                continue
            d = np.sort(r[split]["distances"][f"{metric}_{unit}"])
            ax.plot(d, np.linspace(0, 1, len(d)), color="k" if name == YARDSTICK else f"C{k}", ls="--" if name == YARDSTICK else "-", label=name)
        ax.set_xlim(0, xmax)
        ax.set_title(f"{split}: {metric}")
        ax.set_xlabel(label)
        ax.grid(alpha=0.3)
axes[0, 0].set_ylabel("P(d < ε)")
axes[1, 0].set_ylabel("P(d < ε)")
axes[0, 3].legend(loc="lower right", fontsize=8)
fig.tight_layout()
plt.show()

## 7. Summary

All numbers in one JSON (without the per-pose distances), for the write-up.

In [ ]:
summary = {name: {split: {k: v for k, v in r[split].items() if k != "distances"} | (
               {"likelihood": {k: v for k, v in r[split]["likelihood"].items() if k != "values"}} if r[split].get("likelihood") else {})
               for split in ("test", "train") if split in r} for name, r in reports.items()}
(REPORT_DIR / "summary.json").write_text(json.dumps({"paper": PAPER, "models": summary}, indent=2))
print(f"saved to {REPORT_DIR / 'summary.json'}")